# TissueMC: week-1 GPU smoke test (about 15 minutes)

This notebook checks whether the free GPU tier can support the TissueMC project. **It doesn't run any project code.** It only checks the environment.

**Before you start:** select a T4 GPU.
- **Colab:** *Runtime → Change runtime type → T4 GPU*.
- **Kaggle:** *Settings → Accelerator → GPU T4 x2*. Don't choose P100, because the current CUDA toolkit no longer targets it.

**Then:**
1. Run every cell in order (*Runtime → Run all*).
2. Read the **final JSON block** from the last cell. It summarizes what the free tier supports.

Nothing here needs passwords or API keys, and nothing is uploaded anywhere.

In [ ]:
import json, subprocess, shutil, platform
report = {}

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()

rc, out = sh('nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader')
report['gpu'] = out if rc == 0 else f'NO GPU ({out[:200]})'
rc, out = sh('nvidia-smi')
print(out[:1500])

In [ ]:
rc, out = sh('nvcc --version')
report['nvcc'] = out.splitlines()[-1] if rc == 0 else 'nvcc NOT FOUND'
print(out)

kernel = r'''
#include <cstdio>
__global__ void k(int *c){ if (threadIdx.x == 0) atomicAdd(c, 1); }
int main(){
  int *c; cudaMallocManaged(&c, sizeof(int)); *c = 0;
  k<<<1024, 128>>>(c); cudaError_t e = cudaDeviceSynchronize();
  printf("blocks_counted=%d err=%s\n", *c, cudaGetErrorString(e));
  return (e == cudaSuccess && *c == 1024) ? 0 : 1;
}
'''
open('hello.cu', 'w').write(kernel)
rc, out = sh('nvcc -O2 -arch=sm_75 -lineinfo hello.cu -o hello && ./hello')
report['cuda_compile_and_run'] = 'PASS' if rc == 0 else f'FAIL: {out[:300]}'
print(out)

In [ ]:
# Can we read GPU hardware performance counters? This decides whether the study can use Nsight Compute.
rc, out = sh('grep RmProfilingAdminOnly /proc/driver/nvidia/params')
report['RmProfilingAdminOnly'] = out if out else 'not readable'
ncu = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'
rc, out = sh(f'{ncu} --section SpeedOfLight -c 1 ./hello')
if 'ERR_NVGPUCTRPERM' in out:
    report['ncu'] = 'BLOCKED (ERR_NVGPUCTRPERM: counters need admin)'
elif rc == 0 and ('SM Throughput' in out or 'Duration' in out):
    report['ncu'] = 'WORKS'
else:
    report['ncu'] = f'UNCLEAR rc={rc}: {out[-300:]}'
print(out[-2000:])

In [ ]:
# MCX (pmcx), the established GPU photon simulator, will be our external GPU baseline. Does it install and run here?
rc, out = sh('pip install -q pmcx')
try:
    import numpy as np, pmcx, time
    vol = np.ones((60, 60, 60), dtype='uint8')
    cfg = {'nphoton': 1_000_000, 'vol': vol, 'tstart': 0, 'tend': 5e-9, 'tstep': 5e-9,
           'srcpos': [30, 30, 0], 'srcdir': [0, 0, 1],
           'prop': [[0, 0, 1, 1], [0.01, 10.0, 0.9, 1.37]], 'issrcfrom0': 1}
    t = time.time(); res = pmcx.run(cfg); dt = time.time() - t
    report['pmcx'] = f'WORKS ({dt:.2f}s for 1e6 photons, smoke test only, not a benchmark)'
except Exception as e:
    report['pmcx'] = f'FAIL: {type(e).__name__}: {str(e)[:200]}'
print(report['pmcx'])

In [ ]:
rc, out = sh('nproc'); report['cpu_threads'] = out
report['python'] = platform.python_version()
rc, out = sh('cmake --version'); report['cmake'] = out.splitlines()[0] if rc == 0 else 'not found'
print('===== SMOKE TEST SUMMARY =====')
print(json.dumps(report, indent=2))